# exp012 h2o

## 0. Experiment Metadata

In [21]:
# ========================================
# EXPERIMENT CONFIG
# ========================================

EXP_NAME = "exp012_h2o"
TARGET = "Churn"
ID_COL = "id"

N_SPLITS = 5
SEED = 42

print(f"Running {EXP_NAME}")

Running exp012_h2o


## 1. Imports

In [2]:
# ========================================
# IMPORTS
# ========================================

import os
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

import lightgbm as lgb
import xgboost as xgb

import warnings
warnings.filterwarnings("ignore")

pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',50)

## 2. Reproducibility

In [3]:
# ========================================
# SEED EVERYTHING
# ========================================

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(SEED)

## 3. Load Data

In [4]:
# ========================================
# LOAD DATA
# ========================================

DATA_PATH = "/Users/theojeremiah/Workspace/01_DataScience/Projects/202603_Kaggle_CustomerChurn/data/raw/"

train = pd.read_csv(DATA_PATH + "train.csv")
test = pd.read_csv(DATA_PATH + "test.csv")

print(train.shape, test.shape)
train.head()

(594194, 21) (254655, 20)


,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,No,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,Yes,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


## 4. Quick Sanity Checs (Fast EDA)

In [5]:
# ========================================
# QUICK EDA
# ========================================

print("Target distribution:")
print(train['Churn'].value_counts(normalize=True))

print("\nMissing values (top 10):")
print(train.isnull().mean().sort_values(ascending=False).head(10))

Target distribution:
Churn
No     0.774792
Yes    0.225208
Name: proportion, dtype: float64

Missing values (top 10):
id                  0.0
DeviceProtection    0.0
TotalCharges        0.0
MonthlyCharges      0.0
PaymentMethod       0.0
PaperlessBilling    0.0
Contract            0.0
StreamingMovies     0.0
StreamingTV         0.0
TechSupport         0.0
dtype: float64


## 5. Feature Engineering 

In [6]:
# ========================================
# FEATURE ENGINEERING (EXP006 - COMBINED BEST)
# ========================================

service_cols = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection", 
    "TechSupport", "StreamingTV", "StreamingMovies"
]

def create_features(df):
    df = df.copy()

    # ========================================
    # FINANCIAL (STRONG SIGNAL)
    # ========================================
    if all(col in df.columns for col in ["TotalCharges", "tenure"]):
        df["avg_charge"] = df["TotalCharges"] / (df["tenure"] + 1)

    if all(col in df.columns for col in ["TotalCharges", "MonthlyCharges", "tenure"]):
        df["charge_diff"] = df["TotalCharges"] - (df["MonthlyCharges"] * df["tenure"])

    # ========================================
    # SERVICE USAGE (STRONG SIGNAL)
    # ========================================
    if all(col in df.columns for col in service_cols):
        df["num_services"] = (df[service_cols] == "Yes").sum(axis=1)

    # ========================================
    # BUSINESS LOGIC (VERY STRONG)
    # ========================================
    if "PaymentMethod" in df.columns:
        df["payment_risk"] = (df["PaymentMethod"] == "Electronic check").astype(int)

    if "InternetService" in df.columns:
        df["is_fiber"] = (df["InternetService"] == "Fiber optic").astype(int)

    # ========================================
    # OPTIONAL CLEANUP (IMPORTANT DECISION)
    # ========================================
    # Drop TotalCharges ONLY if using avg_charge + charge_diff
    # if "TotalCharges" in df.columns:
    #     df = df.drop(columns=["TotalCharges"])

    return df


# Apply
train = create_features(train)
test = create_features(test)

## 6. Target Encoding

In [7]:
# ========================================
# TARGET ENCODING (SIMPLE VERSION)
# ========================================

from category_encoders import TargetEncoder

# Get categorical columns from TRAIN
categorical_cols = train.select_dtypes(include="object").columns.tolist()

# Remove target column
categorical_cols = [col for col in categorical_cols if col != TARGET]

# Initialize encoder
encoder = TargetEncoder(cols=categorical_cols)

# Fit on train, transform both
train[categorical_cols] = encoder.fit_transform(train[categorical_cols], train[TARGET])
test[categorical_cols] = encoder.transform(test[categorical_cols])

## 7. Prepare Data

In [8]:
# ========================================
# PREPARE DATA
# ========================================

features = [col for col in train.columns if col not in [TARGET, ID_COL]]

train[TARGET] = train[TARGET].map({"No": 0, "Yes": 1}).astype(int)

X = train[features]
y = train[TARGET]

X_test = test[features]

## 8. Validation Strategy

In [9]:
# ========================================
# CV SETUP
# ========================================

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

## 9. Model Training

In [17]:
import h2o
from h2o.automl import H2OAutoML

h2o.init()

Checking whether there is an H2O instance running at http://localhost:54321. connected.


H2O_cluster_uptime:,21 mins 16 secs
H2O_cluster_timezone:,Asia/Jakarta
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.10
H2O_cluster_version_age:,15 days
H2O_cluster_name:,H2O_from_python_theojeremiah_71hfe8
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,3.721 Gb
H2O_cluster_total_cores:,8
H2O_cluster_allowed_cores:,8
H2O_cluster_status:,"locked, healthy"


In [18]:
# ========================================
# H2O K-FOLD TRAINING (ENHANCED)
# ========================================

oof_preds_h2o = np.zeros(len(train))
test_preds_h2o = np.zeros(len(test))

scores_h2o = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    
    print(f"\n--- Fold {fold} ---")

    X_train, X_valid = X.iloc[tr_idx].copy(), X.iloc[val_idx].copy()
    y_train, y_valid = y.iloc[tr_idx].copy(), y.iloc[val_idx].copy()

    # Combine features + target
    train_fold = X_train.copy()
    train_fold["target"] = y_train

    valid_fold = X_valid.copy()
    valid_fold["target"] = y_valid

    test_fold = X_test.copy()

    # Convert to H2OFrame
    hf_train = h2o.H2OFrame(train_fold)
    hf_valid = h2o.H2OFrame(valid_fold)
    hf_test = h2o.H2OFrame(test_fold)

    # Set target
    hf_train["target"] = hf_train["target"].asfactor()
    hf_valid["target"] = hf_valid["target"].asfactor()

    features = [col for col in hf_train.columns if col != "target"]

    # ========================================
    # H2O AUTOML (UPGRADED)
    # ========================================
    aml = H2OAutoML(
        max_runtime_secs=300,          # ⬆️ more time → better models
        nfolds=5,                     # ⬆️ enable internal CV → better stacking
        balance_classes=True,         # ⬆️ churn imbalance handling
        sort_metric="AUC",
        seed=SEED,

        # 🔥 IMPORTANT: force strong models
        include_algos=["GBM", "GLM", "DRF", "StackedEnsemble"],

        # 🔥 Keep best models only
        keep_cross_validation_predictions=True,
        keep_cross_validation_models=False,
        keep_cross_validation_fold_assignment=False
    )

    aml.train(
        x=features,
        y="target",
        training_frame=hf_train
        # ❌ REMOVE validation_frame → let H2O handle internally
    )

    # 🔥 Use STACKED ENSEMBLE (not just leader blindly)
    leaderboard = aml.leaderboard.as_data_frame()

    print("\nTop Models:")
    print(leaderboard.head())

    best_model = aml.leader  # often already ensemble

    # Validation prediction
    val_pred = best_model.predict(hf_valid).as_data_frame()["p1"].values
    oof_preds_h2o[val_idx] = val_pred

    # Test prediction
    test_pred = best_model.predict(hf_test).as_data_frame()["p1"].values
    test_preds_h2o += test_pred / N_SPLITS

    score = roc_auc_score(y_valid, val_pred)
    scores_h2o.append(score)

    print(f"Fold {fold} AUC: {score:.5f}")


print("\nH2O CV Mean:", np.mean(scores_h2o))
print("H2O CV Std:", np.std(scores_h2o))


--- Fold 0 ---
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
AutoML progress: |███████████████████████████████████████████████████████████████| (done) 100%

Top Models:
                                            model_id       auc   logloss  \
0  StackedEnsemble_AllModels_1_AutoML_6_20260328_...  0.915310  0.299849   
1  StackedEnsemble_AllModels_2_AutoML_6_20260328_...  0.915305  0.299854   
2  StackedEnsemble_AllModels_3_AutoML_6_20260328_...  0.915294  0.299852   
3  StackedEnsemble_BestOfFamily_1_AutoML_6_202603...  0.914683  0.300697   
4                      GBM_3_AutoML_6_20260328_75624  0.914535  0.302267   

      aucpr  mean_per_class_error      rmse       mse  
0  0.751380              0.176484  0.309420  0.095741  
1  0.751397           

/Users/theojeremiah/Workspace/01_DataScience/Projects/202603_Kaggle_CustomerChurn/venv/lib/python3.11/site-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


███████████████████████████████████████████| (done) 100%
stackedensemble prediction progress: |

/Users/theojeremiah/Workspace/01_DataScience/Projects/202603_Kaggle_CustomerChurn/venv/lib/python3.11/site-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


███████████████████████████████████████████| (done) 100%
Fold 0 AUC: 0.91512

--- Fold 1 ---


...

/Users/theojeremiah/Workspace/01_DataScience/Projects/202603_Kaggle_CustomerChurn/venv/lib/python3.11/site-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


███████████████████████████████████████████| (done) 100%
stackedensemble prediction progress: |

/Users/theojeremiah/Workspace/01_DataScience/Projects/202603_Kaggle_CustomerChurn/venv/lib/python3.11/site-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


███████████████████████████████████████████| (done) 100%
Fold 4 AUC: 0.91336

H2O CV Mean: 0.915210745251232
H2O CV Std: 0.0010233682351797242


/Users/theojeremiah/Workspace/01_DataScience/Projects/202603_Kaggle_CustomerChurn/venv/lib/python3.11/site-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


In [19]:
aml.leaderboard

model_id,auc,logloss,aucpr,mean_per_class_error,rmse,mse
StackedEnsemble_AllModels_1_AutoML_10_20260328_81734,0.915166,0.300106,0.750958,0.173129,0.309571,0.095834
StackedEnsemble_AllModels_2_AutoML_10_20260328_81734,0.915158,0.300119,0.750999,0.170033,0.309575,0.0958368
StackedEnsemble_BestOfFamily_1_AutoML_10_20260328_81734,0.914994,0.300235,0.751025,0.173269,0.309647,0.0958811
StackedEnsemble_BestOfFamily_2_AutoML_10_20260328_81734,0.914984,0.300257,0.750977,0.173361,0.309651,0.0958838
StackedEnsemble_BestOfFamily_3_AutoML_10_20260328_81734,0.914981,0.300269,0.750972,0.173957,0.309655,0.0958865
GBM_1_AutoML_10_20260328_81734,0.914491,0.301359,0.750112,0.175629,0.310112,0.0961694
GBM_3_AutoML_10_20260328_81734,0.911648,0.323561,0.728463,0.173873,0.320298,0.102591
GBM_2_AutoML_10_20260328_81734,0.91143,0.320867,0.727647,0.176879,0.319315,0.101962
GLM_1_AutoML_10_20260328_81734,0.908546,0.311108,0.730075,0.177464,0.315277,0.0993997
GBM_5_AutoML_10_20260328_81734,0.901828,0.378196,0.683452,0.174348,0.348258,0.121284


#### prev score 0.91669

## 11. Feature Importance

In [113]:
# ========================================
# FEATURE IMPORTANCE
# ========================================

import matplotlib.pyplot as plt

importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values(by="importance", ascending=False)

importance.head(20)

plt.figure(figsize=(8, 10))
plt.barh(importance["feature"].head(20), importance["importance"].head(20))
plt.gca().invert_yaxis()
plt.title("Top Features")
plt.show()

ValueError: All arrays must be of the same length

## 12. Create Submission

In [22]:
# ========================================
# SUBMISSION (FINAL - ROBUST VERSION)
# ========================================

import os
import pandas as pd
import numpy as np
from datetime import datetime

# ========================================
# CONFIG
# ========================================
OUTPUT_DIR = "/Users/theojeremiah/Workspace/01_DataScience/Projects/202603_Kaggle_CustomerChurn/outputs/submissions/"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ========================================
# FINAL PREDICTION (USE YOUR BEST VERSION)
# ========================================

# Example (choose ONE depending on your pipeline):

# 1. If using single model (H2O only)
final_preds = test_preds_h2o

# 2. If using ensemble (RECOMMENDED)
# final_preds = (
#     0.5 * test_preds_lgb +
#     0.2 * test_preds_xgb +
#     0.3 * test_preds_h2o
# )

# ========================================
# SAFETY CHECKS
# ========================================

# Ensure prediction length matches test
assert len(final_preds) == len(test), "Prediction length mismatch!"

# Clip probabilities (avoid extreme issues)
final_preds = np.clip(final_preds, 0, 1)

# ========================================
# CREATE SUBMISSION
# ========================================
submission = pd.DataFrame({
    ID_COL: test[ID_COL].values,
    TARGET: final_preds
})

# ========================================
# FILE NAMING (WITH TIMESTAMP)
# ========================================
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

file_path = os.path.join(
    OUTPUT_DIR,
    f"{EXP_NAME}_{timestamp}.csv"
)

# ========================================
# SAVE
# ========================================
submission.to_csv(file_path, index=False)

print(f"Submission saved at: {file_path}")
print(submission.head())

Submission saved at: /Users/theojeremiah/Workspace/01_DataScience/Projects/202603_Kaggle_CustomerChurn/outputs/submissions/exp012_h2o_20260328_0826.csv
       id     Churn
0  594194  0.064751
1  594195  0.002397
2  594196  0.109852
3  594197  0.005095
4  594198  0.519584
